<a href="https://colab.research.google.com/github/LorenzoBellomo/InformationRetrieval/blob/main/notebooks/3_NER_EL_PoS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced NLP, phrases and semantics
In this lecture, we focus on Named Entity Recognition, Entity Linking, and Part of Speech Tagging.


In [18]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import requests
import json
import spacy
from spacy import displacy

## Part of Speech (PoS) Tagging


Let's now use spaCy to see the PoS tagging process, and how we can find dependencies between words in a sentence programmatically.

In [20]:
en_model = spacy.load("en_core_web_sm")
sentence = "He was being opposed by her without any reason.\
	    A plan is being prepared by Charles for next project"
processed_sent = en_model(sentence)
for token in processed_sent:
    print(f'{token.text:{10}} {token.tag_:>{10}}\t{spacy.explain(token.tag_):<{50}}')

He                PRP	pronoun, personal                                 
was               VBD	verb, past tense                                  
being             VBG	verb, gerund or present participle                
opposed           VBN	verb, past participle                             
by                 IN	conjunction, subordinating or preposition         
her               PRP	pronoun, personal                                 
without            IN	conjunction, subordinating or preposition         
any                DT	determiner                                        
reason             NN	noun, singular or mass                            
.                   .	punctuation mark, sentence closer                 
	                 _SP	whitespace                                        
A                  DT	determiner                                        
plan               NN	noun, singular or mass                            
is                VBZ	verb, 3rd person singular pre

In [21]:
displacy.render(processed_sent, style="dep", jupyter=True)

## Named Entity Recognition

Now we will load the spacy english model, and test its default NER pipeline on a sample text

In [22]:
en_model = spacy.load("en_core_web_sm") # Loading the English model

text = "Apple is looking at buying U.K. startup for $1 billion"
doc = en_model(text)
displacy.render(doc, style="ent", jupyter=True)

In [23]:
for entity in doc.ents:
    print(entity.text, " - ", entity.label_, " - ", spacy.explain(entity.label_))

Apple  -  ORG  -  Companies, agencies, institutions, etc.
U.K.  -  GPE  -  Countries, cities, states
$1 billion  -  MONEY  -  Monetary values, including unit


As we're able to see, verbs and less stopwords are ignored, while named entities are properly categorized

Let's test it on another, longer, phrase

In [24]:
!wget https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/Leonardo.txt

--2026-03-11 14:09:14--  https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/Leonardo.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 604 [text/plain]
Saving to: ‘Leonardo.txt’

Leonardo.txt        100%[===================>]     604  --.-KB/s    in 0s      

2026-03-11 14:09:14 (22.9 MB/s) - ‘Leonardo.txt’ saved [604/604]



In [25]:
with open("Leonardo.txt", 'r') as txt_file:
  leonardo = txt_file.read()
leonardo

'Leonardo da Vinci was an Italian Renaissance polymath whose areas of interest included invention, painting, sculpting, architecture, science, music, mathematics, engineering, literature, anatomy, geology, astronomy, botany, writing, history, and cartography. \nHe has been variously called the father of palaeontology, ichnology, and architecture, and is widely considered one of the greatest painters of all time. Leonardo is revered for his technological ingenuity. He conceptualised flying machines, a type of armoured fighting vehicle, concentrated solar power, an adding machine, and the double hull.'

In [26]:
doc = en_model(leonardo)
displacy.render(doc, style="ent", jupyter=True)

Something goes wrong here, why is Renaissance labeled as ORG? Let's see the meaning of the various categories

In [27]:
for entity in doc.ents:
  print(entity.text, " - ", entity.label_, " - ", spacy.explain(entity.label_))

Leonardo da Vinci  -  PERSON  -  People, including fictional
Italian  -  NORP  -  Nationalities or religious or political groups
Renaissance  -  ORG  -  Companies, agencies, institutions, etc.
one  -  CARDINAL  -  Numerals that do not fall under another type
Leonardo  -  PERSON  -  People, including fictional


This shows how crucial it is, in some situations, to train your application to your specific need.

### Finetuning NER

Now we download the Anatomical Entity Mention (AnEM) corpus from Github. It is a NER dataset on the biomedical field, stored using the CONLL format.  

In [28]:
!wget https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/AnEM.train
!wget https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/AnEM.test

--2026-03-11 14:09:17--  https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/AnEM.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1220623 (1.2M) [text/plain]
Saving to: ‘AnEM.train’

AnEM.train          100%[===================>]   1.16M  --.-KB/s    in 0.05s   

2026-03-11 14:09:17 (22.8 MB/s) - ‘AnEM.train’ saved [1220623/1220623]

--2026-03-11 14:09:17--  https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/data/AnEM.test
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 772970 (755K) [text/plain

spaCy does not accept files in the CONLL format, so we use its convert function. We need to rename files to avoid name collisions

In [29]:
!mv AnEM.train AnEMTrain
!mv AnEM.test AnEMTest
!spacy convert AnEMTrain -c conll .
!spacy convert AnEMTest -c conll .

ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (2815 documents): AnEMTrain.spacy
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1882 documents): AnEMTest.spacy


**NOW IT IS TIME TO GENERATE THE base_config.cfg file**
INSTRUCTIONS ARE HERE: https://spacy.io/usage/training#quickstart

In [30]:
!wget https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/base_config.cfg
!python -m spacy init fill-config base_config.cfg config.cfg

--2026-03-11 14:09:37--  https://raw.githubusercontent.com/giusprencunipi/IR-Master/main/base_config.cfg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1713 (1.7K) [text/plain]
Saving to: ‘base_config.cfg’

base_config.cfg     100%[===================>]   1.67K  --.-KB/s    in 0s      

2026-03-11 14:09:37 (20.7 MB/s) - ‘base_config.cfg’ saved [1713/1713]

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


For reasons of lecture pacing, I limit the epochs to 2.

In [31]:
!mkdir -p output
#!python -m spacy train config.cfg --output ./output --paths.train ./AnEMTrain.spacy --paths.dev ./AnEMTest.spacy
!python -m spacy train config.cfg --output ./output --paths.train ./AnEMTrain.spacy --paths.dev ./AnEMTest.spacy --training.max_epochs 2

ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     27.87    0.00    0.00    0.00    0.00
  0     200        308.78   1644.46    8.64   13.06    6.45    0.09
  0     400         85.48    866.23   15.94   33.76   10.43    0.16
  1     600        139.98   1002.81   25.36   33.55   20.38    0.25
  1     800        236.84   1008.64   37.06   50.41   29.30    0.37
✔ Saved pipeline to output directory
output/model-last


In [35]:
print(nlp.pipe_names) #standard model pipeline
trained_model = spacy.load("output/model-best")
print(trained_model.pipe_names) #new trained model pipeline
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = trained_model(text)
displacy.render(doc, style="ent", jupyter=True)

['tok2vec', 'morphologizer', 'tagger', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']
['tok2vec', 'ner']


/usr/local/lib/python3.12/dist-packages/spacy/displacy/__init__.py:215: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


As we expect, a general purpose phrase yields no Named Entities

In [36]:
text = "In the posterior (dorsal) cavity, the cranial cavity houses the brain, and the spinal cavity (or vertebral cavity) encloses the spinal cord."
doc = trained_model(text)
displacy.render(doc, style="ent", jupyter=True)

And as expected, the new biological categories appear!

Exercise:
1.   Given the query phrase, Print all the nouns.

## Entity Linking

### Initialize TagMe
Please download your own access key to TagMe's service, according to the slides seen in class

In [ ]:
KEY = "6ab7daea-c174-4254-a9d2-d85f6117bf20-843339462" # this is the key we will be using for REST calls

In [ ]:
TAGME_ENDPOINT = "https://tagme.d4science.org/tagme/tag"
LANG = "en" # Also works in italian and german

Now create the function that will "wrap" the REST call. It needs a textual input

In [ ]:
def query_tagme(text):
    payload = {"text": text, "gcube-token": KEY, "lang": LANG}
    # Now we issue a post HTTP request
    r = requests.post(TAGME_ENDPOINT, payload)
    if r.status_code != 200:
        # this means something went wrong with the query
        raise Exception("Error on text: {}\n{}".format(text, r.text))
    return r.json()

And now we display the result for a simple textual query. The interesting part, for us, is under the key _annotations_.
This will be a list of annotations containing the following fields:
- **spot (string)**: how the anchor appears in the text.
- **start (int)**: the index of the starting character of the anchor.
- **end (int)**: the index of the ending character of the anchor.
- **link_probability (float ∈[𝟎,𝟏])**: number of times that the spot is an anchor in Wikipedia / number of occurrences of the spot in Wikipedia.
- **rho (float ∈[𝟎,𝟏])**: semantic coherency of the entity with respect to the query.
- **id (int)**: the Wikipedia identifier of the page _(https://en.wikipedia.org/?curid=<>)_.
- **title (string)**: title of the Wikipedia page.

In [ ]:
short_text = "Italy will not be competing in the 2022 world cup"
resp = query_tagme(short_text)
resp

{'test': '5',
 'annotations': [{'spot': 'Italy',
   'start': 0,
   'link_probability': 0.4437723457813263,
   'rho': 0.4525856375694275,
   'end': 5,
   'id': 362466,
   'title': 'Italy national football team'},
  {'spot': 'will',
   'start': 6,
   'link_probability': 0.0036389119923114777,
   'rho': 0.06729841977357864,
   'end': 10,
   'id': 32828260,
   'title': 'Will (2011 film)'},
  {'spot': '2022 world cup',
   'start': 35,
   'link_probability': 0.3492063581943512,
   'rho': 0.3398236632347107,
   'end': 49,
   'id': 17742072,
   'title': '2022 FIFA World Cup'}],
 'time': 30,
 'api': 'tag',
 'lang': 'en',
 'timestamp': '2026-03-05T11:33:32'}

### Handle longer texts / filtering noisy annotations

TagME has been designed for handling short texts, but we also have a way to obtain competitive results on longer ones.
This requires modifying the window of spots that are checked by TagME when doing disambiguation.

Now open a new text file with a slightly longer text and annotate it with TagME

In [4]:
with open("Leonardo.txt", 'r') as long_file:
    # the text is not a json object, it is just a plaintext, so just read it regularly with read()
    text = long_file.read()
text

'Leonardo da Vinci was an Italian Renaissance polymath whose areas of interest included invention, painting, sculpting, architecture, science, music, mathematics, engineering, literature, anatomy, geology, astronomy, botany, writing, history, and cartography. \nHe has been variously called the father of palaeontology, ichnology, and architecture, and is widely considered one of the greatest painters of all time. Leonardo is revered for his technological ingenuity. He conceptualised flying machines, a type of armoured fighting vehicle, concentrated solar power, an adding machine, and the double hull.'

Now we will change the tagging function we made before, by adding an optional boolean parameter. If true, this means that the text is long, otherwise it is short and we can avoid changing the window.

In [ ]:
def query_tagme(text, long_text=False):
    payload = {"text": text, "gcube-token": KEY, "lang": LANG}
    if long_text:
        # long_text is by defaul false, but if specified by the user, we set the window size at 5
        payload["long_text"] = 5
    r = requests.post(TAGME_ENDPOINT, payload)
    if r.status_code != 200:
        raise Exception("Error on text: {}\n{}".format(text, r.text))
    return r.json()

But how do we deal with noisy annotations? TagME gives us a "content relevance" score in the form of the **Rho-score**.
We can filter the lowest ranked annotations on relevancy to remove noise. A common threshold for this task is 0.3.

In [ ]:
# Try changing the min_rho parameter and see how it impacts the returned entities
def get_tagme_entities(tagme_response, min_rho=0.3):
    ann = tagme_response["annotations"]
    ann = [a for a in ann if a["rho"] > min_rho] # filter all the annotations with a rho score lower than the threshold
    return [a["title"] for a in ann if "title" in a] # return just the page titles

Now see which entities _disappear_ when filtering

```js
[{'spot': 'Italy',
   'start': 0,
   'link_probability': 0.4437723457813263,
   'rho': 0.4525856375694275,
   'end': 5,
   'id': 362466,
   'title': 'Italy national football team'},
  {'spot': 'will',
   'start': 6,
   'link_probability': 0.0036389119923114777,
   'rho': 0.06729841977357864,
   'end': 10,
   'id': 32828260,
   'title': 'Will (2011 film)'},
  {'spot': '2022 world cup',
   'start': 35,
   'link_probability': 0.3492063581943512,
   'rho': 0.3398236632347107,
   'end': 49,
   'id': 17742072,
   'title': '2022 FIFA World Cup'}]
   ```

In [ ]:
print("BEFORE FILTERING")
resp = query_tagme(text, long_text=True)
before_filtering = [(a["spot"], a["title"], round(a["rho"]*100)/100) for a in resp['annotations'] if "title" in a]
before_filtering

BEFORE FILTERING


[('Leonardo da Vinci', 'Leonardo da Vinci', 0.83),
 ('Vinci', 'Leonardo da Vinci', 0.36),
 ('Italian Renaissance', 'Italian Renaissance', 0.39),
 ('polymath', 'Polymath', 0.52),
 ('interest', 'Attention', 0.11),
 ('invention', 'Invention', 0.21),
 ('painting', 'Painting', 0.22),
 ('sculpting', 'Sculpture', 0.25),
 ('architecture', 'Architecture', 0.28),
 ('science', 'Science', 0.21),
 ('music, mathematics', 'Music and mathematics', 0.44),
 ('engineering', 'Engineering', 0.24),
 ('literature', 'Literature', 0.22),
 ('anatomy', 'Anatomy', 0.29),
 ('geology', 'Geology', 0.31),
 ('astronomy', 'Astronomy', 0.39),
 ('botany', 'Botany', 0.33),
 ('writing', 'Writing', 0.16),
 ('history', 'History', 0.19),
 ('cartography', 'Cartography', 0.33),
 ('father', 'Clergy', 0.09),
 ('palaeontology', 'Paleontology', 0.33),
 ('ichnology', 'Ichnology', 0.53),
 ('architecture', 'Architecture', 0.11),
 ('one', 'Neoplatonism', 0.12),
 ('greatest', 'Greatest!', 0.0),
 ('painters', 'Painting', 0.11),
 ('time',

In [ ]:
print("AFTER FILTERING")
after_filtering = get_tagme_entities(resp)
after_filtering

AFTER FILTERING


['Leonardo da Vinci',
 'Leonardo da Vinci',
 'Italian Renaissance',
 'Polymath',
 'Music and mathematics',
 'Geology',
 'Astronomy',
 'Botany',
 'Cartography',
 'Paleontology',
 'Ichnology',
 'Armoured fighting vehicle',
 'Concentrated solar power',
 'Adding machine']

In [ ]:
print("The annotations that were filtered out are:")
[a for a in before_filtering if a not in after_filtering]

The annotations that were filtered out are:


[('Leonardo da Vinci', 'Leonardo da Vinci', 0.83),
 ('Vinci', 'Leonardo da Vinci', 0.36),
 ('Italian Renaissance', 'Italian Renaissance', 0.39),
 ('polymath', 'Polymath', 0.52),
 ('interest', 'Attention', 0.11),
 ('invention', 'Invention', 0.21),
 ('painting', 'Painting', 0.22),
 ('sculpting', 'Sculpture', 0.25),
 ('architecture', 'Architecture', 0.28),
 ('science', 'Science', 0.21),
 ('music, mathematics', 'Music and mathematics', 0.44),
 ('engineering', 'Engineering', 0.24),
 ('literature', 'Literature', 0.22),
 ('anatomy', 'Anatomy', 0.29),
 ('geology', 'Geology', 0.31),
 ('astronomy', 'Astronomy', 0.39),
 ('botany', 'Botany', 0.33),
 ('writing', 'Writing', 0.16),
 ('history', 'History', 0.19),
 ('cartography', 'Cartography', 0.33),
 ('father', 'Clergy', 0.09),
 ('palaeontology', 'Paleontology', 0.33),
 ('ichnology', 'Ichnology', 0.53),
 ('architecture', 'Architecture', 0.11),
 ('one', 'Neoplatonism', 0.12),
 ('greatest', 'Greatest!', 0.0),
 ('painters', 'Painting', 0.11),
 ('time',

In [ ]:
query_tagme("Maradona visits Mexico")

{'test': '5',
 'annotations': [{'spot': 'Maradona',
   'start': 0,
   'link_probability': 0.15395480394363403,
   'rho': 0.391755610704422,
   'end': 8,
   'id': 8485,
   'title': 'Diego Maradona'},
  {'spot': 'Mexico',
   'start': 16,
   'link_probability': 0.41666820645332336,
   'rho': 0.5231122970581055,
   'end': 22,
   'id': 808402,
   'title': 'Mexico national football team'}],
 'time': 7,
 'api': 'tag',
 'lang': 'en',
 'timestamp': '2026-03-05T11:33:33'}

### TRY OTHER ANNOTATORS: SWAT

TagME is not the only available annotator. There are several more, each one with its own strenghts and weaknesses.
Most of the available annotators are available at [this](https://sobigdata.d4science.org/web/tagme/service-overview) page on the SoBigData Infrastructure

**SWAT** is specifically a salient entity linker, which works best on long, well-constructed texts.
The fields returned are:
- **salience_class (int)**: 1 if the entity is deemed salient, 0 otherwise
- **salience_score (float ∈[𝟎,𝟏])**: the saliency of the enitity in the text (similar to the rho-score in tagme)
- **spans (list)**: list of times where this entity appears, they are described as:
    - *start (int)*: the index of the starting character of the anchor
    - *end (int)*: the index of the ending character of the anchor
- **wiki_id (int)**: the Wikipedia identifier of the page
- **wiki_title (string)**: title of the Wikipedia page

In [30]:
KEY = "6ab7daea-c174-4254-a9d2-d85f6117bf20-843339462" # this is the key we will be using for REST calls

with open("Leonardo.txt", 'r') as long_file:
    # the text is not a json object, it is just a plaintext, so just read it regularly with read()
    text = long_file.read()
text

# this is the new URL of the annotator on the SoBigData Infrastructure
SWAT_ENDPOINT = "https://swat.d4science.org/salience"

# SWAT also requires a title of the content
def query_swat(title, content):
    document = json.dumps({"title": title, "content": content})
    r = requests.post(SWAT_ENDPOINT, data = document, params={'gcube-token': KEY})
    if r.status_code != 200:
        raise Exception("Error on article titled: {}\n{}".format(title, r.text))
    return r.json()["annotations"]

query_swat("Leonardo da Vinci", text)[:7]

[{'salience_class': 1.0,
  'salience_score': 0.9471508264541626,
  'spans': [{'end': 17, 'start': 0}, {'end': 422, 'start': 414}],
  'wiki_id': 18079,
  'wiki_title': 'Leonardo_da_Vinci'},
 {'salience_class': 1.0,
  'salience_score': 0.5190669894218445,
  'spans': [{'end': 32, 'start': 25}],
  'wiki_id': 14532,
  'wiki_title': 'Italy'},
 {'salience_class': 1.0,
  'salience_score': 0.5682003498077393,
  'spans': [{'end': 44, 'start': 33}],
  'wiki_id': 25532,
  'wiki_title': 'Renaissance'},
 {'salience_class': 0.0,
  'salience_score': 0.4803982079029083,
  'spans': [{'end': 65, 'start': 60}],
  'wiki_id': 9630,
  'wiki_title': 'Ecology'},
 {'salience_class': 0.0,
  'salience_score': 0.35197311639785767,
  'spans': [{'end': 77, 'start': 69}],
  'wiki_id': 146738,
  'wiki_title': 'Interest'},
 {'salience_class': 0.0,
  'salience_score': 0.42167073488235474,
  'spans': [{'end': 96, 'start': 87}],
  'wiki_id': 44312,
  'wiki_title': 'Invention'},
 {'salience_class': 1.0,
  'salience_score':

### RELATEDNESS
Ok but now that I have entities, how do I deal with them? I need to know which are similar and which are not.
If we don't see any way of "dealing with the entities", how do we unlock its full potential? How is this method more powerful than dealing with generic words as tokens?

There are several ways in which we can obtain the relatedness of couples of entities.
The main one that is shown in this notebook is by querying TagME itself. TagME has an internal relatedness computation framework, so I can ask TagME itself how close two entities are to one another. This metric is computed directly on the Wikipedia Knowledge Graph.

In [ ]:
# The URL where the relatedness is given
ENDPOINT_RELATEDNESS = "https://tagme.d4science.org/tagme/rel"

# In case I need efficiency I can do batch queries of 100 couples per HTTP call
def query_relatedness(e1, e2):
    # Entities require underscores in-place of the spaces. The space is between entity one and entity two
    tt = e1.replace(" ", "_") + " " + e2.replace(" ", "_")
    payload = {"tt": tt, "gcube-token": KEY, "lang": LANG}
    r = requests.post(ENDPOINT_RELATEDNESS, payload)
    if r.status_code != 200:
        raise Exception("Error on relatedness computation: {}\n{}".format(tt, r.text))
    return r.json()

Now let's test the relatedness of three entities.
Two are closely related to one-another (biology and biotechnology).
The last one is completely out of context.

In [ ]:
first = query_relatedness("Biology", "Biotechnology")
second = query_relatedness("Barack Obama", "Biotechnology")
thirds = query_relatedness("Barack Obama", "Biology")
print(first['result'])
print(second['result'])
print(thirds['result'])

[{'couple': 'Biology Biotechnology', 'rel': 0.6070536971092224}]
[{'couple': 'Barack_Obama Biotechnology', 'rel': 0.23863035440444946}]
[{'couple': 'Barack_Obama Biology', 'rel': 0.16491788625717163}]


Let us now go back to that sentence we used for the PoS tagging, and see what TagME would find

In [ ]:
sentence

'He was being opposed by her without any reason.\t    A plan is being prepared by Charles for next project'

In [ ]:
resp = query_tagme(sentence)
[(a["spot"], a["title"], a["rho"]) for a in resp["annotations"]]

[('reason', 'Reason', 0.09262093156576157),
 ('plan', 'Finger protocol', 0.07667145878076553),
 ('Charles', 'Charles Koch', 0.0429387167096138),
 ('for next', 'For loop', 0.07545013725757599),
 ('next', 'Next plc', 0.001524612889625132),
 ('project', 'Psychological projection', 0.04952556639909744)]

As expected, the results are not exceptional, this is because the phrase has no entities. Let's see a visual representation of the PoS tagging, which works much better in this context.

Exercise: Given a phrase, make a query on TagME, and filter out all the annotations with rho score > 0.2

In [ ]:
phrase ="testing phrase"
resp = query_tagme(phrase, long_text=False)
def my_filter(annotations):
  return [a for a in annotations if a["rho"] > 0.2]
my_filter(resp["annotations"])

[{'spot': 'testing',
  'start': 0,
  'link_probability': 0.004709714557975531,
  'rho': 0.22733774781227112,
  'end': 7,
  'id': 241717,
  'title': 'Clinical trial'},
 {'spot': 'phrase',
  'start': 8,
  'link_probability': 0.014681450091302395,
  'rho': 0.23232361674308777,
  'end': 14,
  'id': 44975,
  'title': 'Phrase'}]

In [ ]:
phrase ="testing phrase"
resp = query_tagme(phrase, long_text=False)
filtered_annotations = [a for a in resp["annotations"] if a["rho"] > 0.2]
filtered_annotations

[{'spot': 'testing',
  'start': 0,
  'link_probability': 0.004709714557975531,
  'rho': 0.22733774781227112,
  'end': 7,
  'id': 241717,
  'title': 'Clinical trial'},
 {'spot': 'phrase',
  'start': 8,
  'link_probability': 0.014681450091302395,
  'rho': 0.23232361674308777,
  'end': 14,
  'id': 44975,
  'title': 'Phrase'}]

In [ ]:
filtered_annotations = []
phrase ="testing phrase"
resp = query_tagme(phrase, long_text=False)
for annotation in resp["annotations"]:
  if annotation["rho"] > 0.2:
    filtered_annotations.append(annotation)
filtered_annotations

[{'spot': 'testing',
  'start': 0,
  'link_probability': 0.004709714557975531,
  'rho': 0.22733774781227112,
  'end': 7,
  'id': 241717,
  'title': 'Clinical trial'},
 {'spot': 'phrase',
  'start': 8,
  'link_probability': 0.014681450091302395,
  'rho': 0.23232361674308777,
  'end': 14,
  'id': 44975,
  'title': 'Phrase'}]

Next exercise, given the next two phrases defined below, make a query on TagME for both of them (in Italian and with rho >= 0.35). Then:
- for each pairs of annotations (one from the first phrase, one from the second) print their relatedness.
- find the pair that maximises this relatedness.

In [ ]:
phrase1 = "Diego Armando Maradona è stato un calciatore, allenatore di calcio e dirigente sportivo \
      argentino di ruolo centrocampista offensivo, campione del mondo nel 1986 e vicecampione del mondo nel 1990 con la \
      nazionale argentina. Soprannominato El Pibe de Oro, è considerato uno dei più grandi calciatori di tutti i tempi. \
      In una carriera da professionista più che ventennale militò nell'Argentinos Juniors, nel Boca Juniors, nel Barcellona, \
       nel Napoli, nel Siviglia e nel Newell's Old Boys."
phrase2 = "Ottavio Bianchi è un ex calciatore e allenatore di calcio italiano, di ruolo centrocampista. \
        Nel 1985 giunse al Milan, dove vinse il primo scudetto della storia partenopea nel campionato 1986-1987, \
        conquistando nella stessa stagione anche la Coppa Italia. Nella stagione successiva Bianchi raggiunse la finale \
        di Coppa Italia (persa contro la Sampdoria) e vinse la Coppa UEFA."
LANG = "it"
def my_filter(response):
  return [a for a in response["annotations"] if a["rho"] > 0.35]

annotations1 = my_filter(query_tagme(phrase1, long_text=True))
annotations2 = my_filter(query_tagme(phrase2, long_text=True))

In [ ]:
max_rel = 0
max_rel_pair = ""
for a1 in annotations1:
  for a2 in annotations2:
    rel_result = query_relatedness(a1["title"], a2["title"])["result"][0]
    if rel_result["rel"] > max_rel:
      max_rel_pair = rel_result["couple"]
      max_rel = rel_result["rel"]
print(max_rel, " - ", max_rel_pair)

# ESERCIZIO SpaCy + SWAT

Given two phrases in Italian (phrase1 and phrase2, see Colab), use SWAT to annotate both phrases
Extract the linked entity identifiers (e.g., Wikipedia titles/URIs) from SWAT output
Represent each entity as a vector using a pretrained embedding model (e.g., spaCy vectors)
Compute pairwise cosine similarity between all entity pairs across the two phrases
Report all similarities and the maximum pair

In [3]:
# --- Install spaCy (if needed) ---
!pip -q install spacy requests simplejson

import json
import requests
import spacy

In [4]:
# -------------------------
# 1) Load a spaCy model WITH VECTORS
# -------------------------
# Recommended for Italian if available:
#   it_core_news_md  or  it_core_news_lg
#
# In Colab, you can download like:
# !python -m spacy download it_core_news_md
#
# If you only have *_sm, similarity will be weak/unreliable.

!python -m spacy download it_core_news_md

MODEL_NAME = "it_core_news_md"  # change to it_core_news_lg if you have it
nlp = spacy.load(MODEL_NAME)

# Quick sanity check: do we have vectors?
if nlp.vocab.vectors_length == 0:
    print("WARNING: This spaCy model has no static word vectors.")
    print("   Similarity scores may be poor. Prefer it_core_news_md or it_core_news_lg.")
else:
    print(f"Loaded {MODEL_NAME} with vectors_length={nlp.vocab.vectors_length}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 13.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loaded it_core_news_md with vectors_length=300


In [15]:
# -------------------------
# 2) SWAT query utilities
# -------------------------

KEY = "6ab7daea-c174-4254-a9d2-d85f6117bf20-843339462"  # <-- your gcube-token
SWAT_ENDPOINT = "https://swat.d4science.org/salience"

def query_swat(title: str, content: str):
    """Query SWAT and return the list of annotations."""
    payload = json.dumps({"title": title, "content": content})
    r = requests.post(SWAT_ENDPOINT, data=payload, params={"gcube-token": KEY})
    if r.status_code != 200:
        raise Exception(f"SWAT error for '{title}' (HTTP {r.status_code}):\n{r.text}")
    data = r.json()
    if "annotations" not in data:
        raise Exception(f"Unexpected SWAT response keys: {list(data.keys())}")
    return data["annotations"]

def pick_first_existing(d: dict, keys: list[str]):
    for k in keys:
        if k in d and d[k]:
            return d[k]
    return None

def extract_entity_id(ann: dict):
    """
    Extract a stable entity identifier from one SWAT annotation.
    SWAT JSON fields can vary; we try common ones.
    Returns: (entity_id, surface_form, score_for_ranking)
    """

    #in case annotations are not known, you can guess them and look for the first one
    '''entity_id = pick_first_existing(
        ann,
        keys=[
            "wiki_title", "wikipediaTitle", "title", "entityTitle", "entity",
            "uri", "url", "wikipediaUrl", "wikipediaURI"
        ],
    )'''
    entity_id = ann['wiki_title']

    surface = pick_first_existing(ann, keys=["spot", "mention", "surface", "text", "anchor"]) #in this implementation, none of these exists -> surface = None
    #score = pick_first_existing(ann, keys=["salience", "salience_score", "confidence", "score", "rho"])
    score = ann['salience_score']

    try:
        score = float(score) if score is not None else None
    except Exception:
        score = None

    # Fallback: if we can't find a linked ID, use the surface form (less ideal)
    if entity_id is None and surface is not None:
        entity_id = surface

    return entity_id, surface, score

def extract_entities(annotations, top_k=10):
    """
    Convert annotations into a clean list of entities.
    Keeps top_k by score if available; otherwise keeps original order.
    """
    ents = []
    for ann in annotations:
        entity_id, surface, score = extract_entity_id(ann)
        if entity_id is None:
            continue
        ents.append({"entity_id": entity_id, "surface": surface, "score": score})

    # If there is any score, rank by it (descending). Otherwise keep as-is.
    if any(e["score"] is not None for e in ents):
        ents.sort(key=lambda x: (x["score"] is None, -(x["score"] or -1e9)))
    return ents[:top_k]

In [6]:
# -------------------------
# 3) Relatedness via spaCy similarity
# -------------------------

def spacy_similarity(text1: str, text2: str) -> float:
    """
    Compute semantic relatedness using spaCy's Doc similarity (cosine over vectors).
    """
    doc1 = nlp(text1)
    doc2 = nlp(text2)
    return float(doc1.similarity(doc2))

def solve_swat_spacy_relatedness(phrase1: str, phrase2: str, top_k=10):
    # SWAT annotations
    anns1 = query_swat("phrase1", phrase1)
    anns2 = query_swat("phrase2", phrase2)

    print("ANNOTATIONS for Phrase1", anns1)

    ents1 = extract_entities(anns1, top_k=top_k)
    ents2 = extract_entities(anns2, top_k=top_k)

    print("=== Phrase 1 ===")
    print(phrase1)
    print("\nEntities (top-k):")
    for e in ents1:
        #!r means apply repr() to represent the string
        print(f"- {e['entity_id']!r} (surface={e['surface']!r}, score={e['score']})")

    print("\n=== Phrase 2 ===")
    print(phrase2)
    print("\nEntities (top-k):")
    for e in ents2:
        print(f"- {e['entity_id']!r} (surface={e['surface']!r}, score={e['score']})")

    if not ents1 or not ents2:
        print("\n Not enough entities to compare. Try longer phrases or increase top_k.")
        return

    print("\n=== Pairwise relatedness (spaCy similarity) ===")
    best_score = -1.0
    best_pair = None

    for e1 in ents1:
        for e2 in ents2:
            # Use entity_id as the representation text (in our case, wiki title)
            s = spacy_similarity(e1["entity_id"], e2["entity_id"])
            print(f"[{s: .4f}] ({e1['entity_id']})  <->  ({e2['entity_id']})")
            if s > best_score:
                best_score = s
                best_pair = (e1, e2)

    print("\n=== Best pair (maximum relatedness) ===")
    e1, e2 = best_pair
    print(f"Score: {best_score:.4f}")
    print(f"Phrase1 entity: {e1['entity_id']}  (surface={e1['surface']}, score={e1['score']})")
    print(f"Phrase2 entity: {e2['entity_id']}  (surface={e2['surface']}, score={e2['score']})")

In [33]:
# -------------------------
# 4) Run on your two phrases
# -------------------------

phrase1 = "Diego Armando Maradona è stato un calciatore, allenatore di calcio e dirigente sportivo \
      argentino di ruolo centrocampista offensivo, campione del mondo nel 1986 e vicecampione del mondo nel 1990 con la \
      nazionale argentina. Soprannominato El Pibe de Oro, è considerato uno dei più grandi calciatori di tutti i tempi. \
      In una carriera da professionista più che ventennale militò nell'Argentinos Juniors, nel Boca Juniors, nel Barcellona, \
       nel Napoli, nel Siviglia e nel Newell's Old Boys."
phrase2 = "Ottavio Bianchi è un ex calciatore e allenatore di calcio italiano, di ruolo centrocampista. \
        Nel 1985 giunse al Milan, dove vinse il primo scudetto della storia partenopea nel campionato 1986-1987, \
        conquistando nella stessa stagione anche la Coppa Italia. Nella stagione successiva Bianchi raggiunse la finale \
        di Coppa Italia (persa contro la Sampdoria) e vinse la Coppa UEFA."

phrase1 = "Leonardo da Vinci dipinse la Gioconda a Firenze."
phrase2 = "Il Rinascimento italiano influenzò l'arte in Europa."

solve_swat_spacy_relatedness(phrase1, phrase2, top_k=8)

ANNOTATIONS for Phrase1 [{'salience_class': 1.0, 'salience_score': 0.617630124092102, 'spans': [{'end': 17, 'start': 0}], 'wiki_id': 18079, 'wiki_title': 'Leonardo_da_Vinci'}, {'salience_class': 0.0, 'salience_score': 0.23072342574596405, 'spans': [{'end': 37, 'start': 26}], 'wiki_id': 1924249, 'wiki_title': 'La_Gioconda_(opera)'}, {'salience_class': 0.0, 'salience_score': 0.14614789187908173, 'spans': [{'end': 47, 'start': 40}], 'wiki_id': 11525, 'wiki_title': 'Florence'}]
=== Phrase 1 ===
Leonardo da Vinci dipinse la Gioconda a Firenze.

Entities (top-k):
- 'Leonardo_da_Vinci' (surface=None, score=0.617630124092102)
- 'La_Gioconda_(opera)' (surface=None, score=0.23072342574596405)
- 'Florence' (surface=None, score=0.14614789187908173)

=== Phrase 2 ===
Il Rinascimento italiano influenzò l'arte in Europa.

Entities (top-k):
- 'Renaissance' (surface=None, score=0.5365899801254272)
- 'Europa_(web_portal)' (surface=None, score=0.21063563227653503)

=== Pairwise relatedness (spaCy similar

/tmp/ipykernel_795/1437422049.py:11: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  return float(doc1.similarity(doc2))
